# L13b Lab: A Software Development Kit for the FRED API
Natural gas is the fuel and the main feedstock of many chemical plants. For example, an ammonia plant makes its hydrogen from natural gas. The price of natural gas is a large part of the cost of running such a plant.

How has the price of natural gas changed over the last ten years? The Federal Reserve Bank of St. Louis publishes this price, and hundreds of thousands of other economic time series, in a database called [FRED (Federal Reserve Economic Data)](https://fred.stlouisfed.org). FRED has a RESTful API.

We'll explore this question using a small software development kit (SDK) for three endpoints of the FRED API. You will complete the SDK. You will write the method that builds the URL of one endpoint, and the handler that turns the response of that endpoint into a table.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * __Call an API that requires a key:__ FRED identifies each client by an API key that the client sends in the query string of every request. We'll read the key from a configuration file, which keeps the key out of the notebook.
> * __Build a URL with a query string:__ Each FRED endpoint takes its inputs as `key=value` parameters in the query string of the URL. You will write the method that puts the parameters of the observations endpoint into a URL.
> * __Convert a JSON response into data:__ FRED sends the date and the value of each observation as text, and sends a period when it has no value. You will write a handler that returns the observations as a table of dates and numbers.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this lab's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [ ]:
# Load packages and paths -
include(joinpath(@__DIR__, "Include.jl")); # @__DIR__ locates this notebook's setup file

# Choose one theme; after changing it, rerun this cell and the plotting cells -
theme(:default)   # light background
# theme(:dark)   # dark background

In addition to the Julia standard libraries, we use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), [the `HTTP.jl` package](https://juliaweb.github.io/HTTP.jl/stable/) for the HTTP calls, [Plots.jl](https://docs.juliaplots.org/stable/) for the figures, and [DataFrames.jl](https://dataframes.juliadata.org/stable/) with [PrettyTables.jl](https://ronisbr.github.io/PrettyTables.jl/stable/) for the tables.

The supplied code of the SDK is in four files in the [src](src) folder:

* [src/Types.jl](src/Types.jl) holds the endpoint models, one type for each endpoint that we call.
* [src/Factory.jl](src/Factory.jl) holds the factory methods that construct the URL of a request.
* [src/Network.jl](src/Network.jl) holds the functions that make the HTTP calls.
* [src/Handler.jl](src/Handler.jl) holds the handlers, the functions that process a response.

Your code goes in [src/Compute.jl](src/Compute.jl). After each edit, save the file and re-run the setup cell above, which reloads your code.

### API key
FRED requires an __API key__, a string of 32 characters that identifies the client. Each user of the FRED API must use their own key. To get your key and store it for this lab:

1. Create a free account at [fredaccount.stlouisfed.org](https://fredaccount.stlouisfed.org/apikeys), and request an API key.
2. Copy the file [conf/Configuration-template.toml](conf/Configuration-template.toml) to a new file named `Configuration.toml` in the same folder.
3. In `Configuration.toml`, paste your key between the quotes on the `apikey` line, and save the file.

> __Why a separate file?__ An API key is like a password. A key that is typed into a notebook goes to everyone who gets a copy of the notebook. The configuration file stays on your computer.

[The `read_configuration(...)` function](src/Utility.jl) reads the file and checks that it holds a key. We save the base URL of the FRED API in the `baseurl::String` variable, and your key in the `apikey::String` variable:

In [ ]:
connection = read_configuration(joinpath(CHEME5800_L13B_CONFIG, "Configuration.toml")); # the [connection] section of the file
baseurl = connection["baseurl"]; # base URL of the FRED API
apikey = connection["apikey"]; # your FRED API key; the semicolon keeps the key off the screen

___
## Task 1: Search FRED for a Data Series
In this task, we search FRED for the series that holds the price of natural gas.

Each series in FRED has an identifier, a short code such as `GNPCA`. [The `fred/series/search` endpoint](https://fred.stlouisfed.org/docs/api/fred/series_search.html) returns the series that match a search text. In the United States, the reference price of natural gas is the price at the Henry Hub, a pipeline junction in Louisiana.

Let's build [a `MyFredSeriesSearchEndpointModel` instance](src/Types.jl) that holds your key, the search text, and the number of series that we want:

In [ ]:
search_model = MyFredSeriesSearchEndpointModel(apikey = apikey, search_text = "Henry Hub natural gas spot price", limit = 5);

Next, we construct the URL of the request with [a `build(...)` method](src/Factory.jl). The URL holds your key, so we do not print it. [The `hidekey(...)` function](src/Utility.jl) returns a copy of the URL that we can show:

In [ ]:
search_url = build(baseurl, search_model); # the URL that we call; it holds your key
hidekey(search_url, apikey) # a copy of the URL with the key replaced

__What do we see?__ The path `/fred/series/search` names the endpoint. The query string after the `?` holds four parameters joined by `&`: the search text, the limit, the key, and the file type. Each `%20` in the search text is an encoded space, because a URL cannot hold a space.

Now, let's make the call. We call the `MyFredSeriesSearchEndpointModel` type as a function with the URL, and pass [the `process_search_response_dataframe(...)` function](src/Handler.jl) through the `handler` keyword argument:

In [ ]:
search_table = MyFredSeriesSearchEndpointModel(search_url, handler = process_search_response_dataframe); # a table with one row per series
pretty_table(search_table[:, [:id, :title, :frequency, :units]]; backend = :text, display_size = (-1, -1),
    table_format = TextTableFormat(borders = text_table_borders__compact));

Each row is one series, with its identifier, title, frequency, and units. FRED holds the Henry Hub price at more than one frequency. We'll use the daily series, which has the identifier `DHHNGSP`.

___
## Task 2: Download the Description of a Series
In this task, we download the description of the Henry Hub series. Then we send a request that fails, and read the error that FRED returns.

[The `fred/series` endpoint](https://fred.stlouisfed.org/docs/api/fred/series.html) returns the description of one series. We build [a `MyFredSeriesEndpointModel` instance](src/Types.jl) with the identifier, construct the URL, and make the call. This time we do not pass a handler. The default handler parses the JSON text into a dictionary:

In [ ]:
series_model = MyFredSeriesEndpointModel(apikey = apikey, series_id = "DHHNGSP");
series_url = build(baseurl, series_model); # the URL that we call; it holds your key
result_series = MyFredSeriesEndpointModel(series_url) # the default handler returns a dictionary

The `seriess` key (FRED spells this key with a double s) holds a list with one entry, the description of our series. Let's print five of its fields:

In [ ]:
description = result_series["seriess"][1]; # the first, and only, entry of the list
println("Title: ", description["title"]);
println("Units: ", description["units"]);
println("Frequency: ", description["frequency"]);
println("First observation: ", description["observation_start"]);
println("Last observation: ", description["observation_end"]);

__What do we see?__ The price is in dollars per million British thermal units (BTU), a unit of energy. FRED has daily observations of this price since January 1997.

### A request that fails
What happens when we ask for a series that does not exist? Let's request the identifier `NOTASERIES`. The `try` block makes the call. When the call throws an error, the `catch` block prints the message of the error:

In [ ]:
bad_model = MyFredSeriesEndpointModel(apikey = apikey, series_id = "NOTASERIES"); # FRED has no series with this identifier
bad_url = build(baseurl, bad_model);
try
    MyFredSeriesEndpointModel(bad_url); # this call fails
catch exception
    println(exception.msg); # the message that our SDK wrote
end

__What do we see?__ FRED answered with a `4xx` status code, which says that the problem is with the request that the client sent. FRED also described the problem in the body of the response. [The private `_http_get_call_with_url(...)` function](src/Network.jl) checked the status code, read the description from the body, and put both into the error message.

___
## Task 3: Complete the SDK for the Observations Endpoint
In this task, you will write the two pieces of the SDK that are not written yet. Then we'll use your code to download the price of natural gas.

[The `fred/series/observations` endpoint](https://fred.stlouisfed.org/docs/api/fred/series_observations.html) returns the observations of a series, which are its data values. [The `MyFredSeriesObservationsEndpointModel` type](src/Types.jl) holds the inputs of the endpoint: the key, the series identifier, the first and last date, the frequency, and the units.

Open [`src/Compute.jl`](src/Compute.jl) and complete its two functions. A TODO comment marks each of the three lines you write:

> __Implementation tasks__
>
> * __TODO 1:__ In [the `build(...)` method](src/Compute.jl), write the URL string of the observations endpoint, with its seven query parameters.
> * __TODO 2:__ In [the `process_observations_response_dataframe(...)` function](src/Compute.jl), convert the date of an observation from text to a `Date`.
> * __TODO 3:__ In the same function, convert the value of an observation from text to a `Float64`.

Save the file, re-run the setup cell at the top of the notebook, and then run the check below. The untouched file throws an error.

### Check: URL and handler
The check uses the example key from the FRED documentation and a short response that we wrote by hand. It does not call FRED:

In [ ]:
@testset "FRED observations endpoint" begin

    # TODO 1: the URL has the base, the path, and the seven parameters, in any order -
    check_model = MyFredSeriesObservationsEndpointModel(apikey = "abcdefghijklmnopqrstuvwxyz123456", series_id = "GNPCA",
        observation_start = "2000-01-01", observation_end = "2010-12-31", frequency = "a", units = "pch");
    check_url = build("https://api.stlouisfed.org", check_model);
    pieces = split(check_url, "?"); # the text before the question mark, and the text after it
    @test length(pieces) == 2
    @test pieces[1] == "https://api.stlouisfed.org/fred/series/observations"
    @test Set(split(pieces[end], "&")) == Set(["series_id=GNPCA", "observation_start=2000-01-01", "observation_end=2010-12-31",
        "frequency=a", "units=pch", "api_key=abcdefghijklmnopqrstuvwxyz123456", "file_type=json"])

    # TODO 2 and TODO 3: three observations; FRED has no value for the second observation -
    check_response = """{"observations": [{"date": "2025-01-01", "value": "3.65"},
        {"date": "2025-01-02", "value": "."}, {"date": "2025-01-03", "value": "4"}]}""";
    check_table = L13bObservations.process_observations_response_dataframe(MyFredSeriesObservationsEndpointModel, check_response);
    @test names(check_table) == ["date", "value"]
    @test check_table.date == [Date(2025, 1, 1), Date(2025, 1, 2), Date(2025, 1, 3)]
    @test isequal(check_table.value, [3.65, missing, 4.0])
end;

### Download the daily price
Now, let's use your code. We ask for the daily Henry Hub price in 2025. The `frequency = "d"` parameter asks for daily observations, and the `units = "lin"` parameter asks for the price itself, with no transformation:

In [ ]:
daily_model = MyFredSeriesObservationsEndpointModel(apikey = apikey, series_id = "DHHNGSP",
    observation_start = "2025-01-01", observation_end = "2025-12-31", frequency = "d", units = "lin");
daily_url = build(baseurl, daily_model); # your build method
daily_table = MyFredSeriesObservationsEndpointModel(daily_url,
    handler = L13bObservations.process_observations_response_dataframe); # your handler
pretty_table(first(daily_table, 8); backend = :text, display_size = (-1, -1),
    table_format = TextTableFormat(borders = text_table_borders__compact)); # the first eight observations

The first row has no value, because the market was closed on New Year's Day. Let's count the missing values, and compute the average price over the days that have a value:

In [ ]:
number_of_missing = count(ismissing, daily_table.value); # the number of days with no value
average_price = mean(skipmissing(daily_table.value)); # skipmissing(...) leaves the missing values out of the average
println("Observations: $(nrow(daily_table)), missing: $(number_of_missing)");
println("Average price in 2025: $(round(average_price, digits = 2)) dollars per million BTU");

### Download ten years of monthly prices
The query parameters can also ask the server to do work for us. With `frequency = "m"`, FRED averages the daily prices of each month, and sends one observation per month.

Let's download the monthly price of natural gas for the ten years from 2016 to 2025. We also download the monthly price of West Texas Intermediate (WTI) crude oil, the reference price of crude oil in the United States. The WTI series has the identifier `DCOILWTICO`:

In [ ]:
# the monthly price of natural gas (units: dollars per million BTU) -
gas_model = MyFredSeriesObservationsEndpointModel(apikey = apikey, series_id = "DHHNGSP",
    observation_start = "2016-01-01", observation_end = "2025-12-31", frequency = "m", units = "lin");
gas_url = build(baseurl, gas_model);
gas_table = MyFredSeriesObservationsEndpointModel(gas_url, handler = L13bObservations.process_observations_response_dataframe);

# the monthly price of WTI crude oil (units: dollars per barrel) -
oil_model = MyFredSeriesObservationsEndpointModel(apikey = apikey, series_id = "DCOILWTICO",
    observation_start = "2016-01-01", observation_end = "2025-12-31", frequency = "m", units = "lin");
oil_url = build(baseurl, oil_model);
oil_table = MyFredSeriesObservationsEndpointModel(oil_url, handler = L13bObservations.process_observations_response_dataframe);

println("Months of natural gas prices: $(nrow(gas_table)), months of crude oil prices: $(nrow(oil_table))");

### Visualize the prices
Let's draw the two monthly prices:

In [ ]:
let
    # the price of natural gas -
    p1 = plot(gas_table.date, gas_table.value, lw = 3, c = :dodgerblue3, label = "Henry Hub natural gas");
    ylabel!(p1, "Price (USD per million BTU)");

    # the price of crude oil -
    p2 = plot(oil_table.date, oil_table.value, lw = 3, c = :red, label = "WTI crude oil");
    ylabel!(p2, "Price (USD per barrel)");

    # labels and style: the gray panel is see-through, so the figure works in both themes -
    for p ∈ [p1, p2]
        xlabel!(p, "Month");
        plot!(p, background_color_inside = RGBA(0.5, 0.5, 0.5, 0.12), framestyle = :box, fg_legend = :transparent,
            legend = :topleft); # both prices start low, so the top left corner is empty
    end
    plot(p1, p2, layout = (1, 2), size = (1000, 420), bottom_margin = 5Plots.mm, left_margin = 5Plots.mm)
end

__What do we see?__ The monthly price of natural gas ranged from about 1.5 to about 8.8 dollars per million BTU over the ten years. Both prices fell in 2020 and peaked in 2022. A plant that buys natural gas saw its feedstock cost change by a factor of more than five.

___
## Summary
In this lab, we built an SDK for three endpoints of the FRED API, and used it to download ten years of natural gas and crude oil prices.

> __Key Takeaways:__
>
> * __An API key identifies the client:__ FRED requires a key in the query string of every request. We kept the key in a configuration file, and hid the key whenever we showed a URL.
> * __The query string carries the inputs of an endpoint:__ We built each URL from a base, a path, and `key=value` parameters joined by `&`. Changing the `frequency` parameter made FRED average the daily prices into monthly prices.
> * __A handler turns text into data:__ FRED sent each date and each value as text, with a period for a missing value. Your handler returned a table with a `Date` column and a `Float64` column, with `missing` where FRED had no value.

We can now change the series identifier, the dates, and the frequency to download other series in FRED.
___